# ML Trading System - Quick Start Demo

This notebook demonstrates the basic usage of the ML Trading System.

## Overview
1. Data Collection
2. Feature Engineering
3. Label Generation (Triple Barrier)
4. Data Exploration
5. Next Steps

In [ ]:
# Add parent directory to path
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import our modules
from src.config import get_default_config
from src.data_collector import DataCollector
from src.feature_engineer import FeatureEngineer
from src.label_generator import LabelGenerator

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Configuration

In [ ]:
# Get default configuration
config = get_default_config()

# Customize for demo (use fewer tickers and shorter time period)
config.tickers = ['SPY', 'QQQ', 'IWM', 'TLT']
config.start_date = '2020-01-01'
config.end_date = '2023-12-31'

print("📋 Configuration:")
print(f"   Tickers: {', '.join(config.tickers)}")
print(f"   Date range: {config.start_date} to {config.end_date}")
print(f"   Profit target: {config.profit_target:.1%}")
print(f"   Stop loss: {config.stop_loss:.1%}")
print(f"   Max holding period: {config.max_holding_period} days")

## 2. Data Collection

In [ ]:
# Initialize collector
collector = DataCollector(config)

# Fetch data
data = collector.fetch_data()

# Show summary
summary = collector.get_data_summary(data)
print("\n📊 Data Summary:")
print(summary)

In [ ]:
# Visualize price history
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, (ticker, df) in enumerate(data.items()):
    axes[i].plot(df.index, df['Close'])
    axes[i].set_title(f'{ticker} Price History')
    axes[i].set_xlabel('Date')
    axes[i].set_ylabel('Price ($)')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
# Initialize feature engineer
engineer = FeatureEngineer(config)

# Create features for all tickers
processed_data = {}
for ticker, df in data.items():
    processed_data[ticker] = engineer.create_features(df)

# Show feature names
feature_names = engineer.get_feature_names(processed_data['SPY'])
print(f"\n🔧 Created {len(feature_names)} features:")
for i, name in enumerate(feature_names[:15], 1):
    print(f"   {i}. {name}")
print(f"   ... and {len(feature_names) - 15} more")

In [ ]:
# Visualize some features for SPY
spy_df = processed_data['SPY'].dropna()

fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# Price with moving averages
axes[0].plot(spy_df.index, spy_df['Close'], label='Close', linewidth=2)
axes[0].plot(spy_df.index, spy_df['ma_20'], label='MA 20', linestyle='--')
axes[0].plot(spy_df.index, spy_df['ma_50'], label='MA 50', linestyle='--')
axes[0].set_title('SPY: Price and Moving Averages')
axes[0].set_ylabel('Price ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# RSI
axes[1].plot(spy_df.index, spy_df['rsi_14'])
axes[1].axhline(70, color='r', linestyle='--', alpha=0.5, label='Overbought')
axes[1].axhline(30, color='g', linestyle='--', alpha=0.5, label='Oversold')
axes[1].set_title('SPY: RSI (14)')
axes[1].set_ylabel('RSI')
axes[1].set_ylim(0, 100)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Volatility
axes[2].plot(spy_df.index, spy_df['volatility_20d'])
axes[2].set_title('SPY: 20-Day Realized Volatility')
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Volatility')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Triple Barrier Labeling

In [ ]:
# Initialize label generator
label_gen = LabelGenerator(config)

# Generate labels for all tickers
labeled_data = {}
for ticker, df in processed_data.items():
    labeled_data[ticker] = label_gen.create_labels(df)

# Show statistics for SPY
stats = label_gen.get_label_statistics(labeled_data['SPY'])
print("\n🏷️  SPY Label Statistics:")
print(f"   Total samples: {stats['total_samples']:,}")
print(f"   Positive (trade): {stats['positive_count']:,} ({stats['positive_pct']:.1f}%)")
print(f"   Negative (no trade): {stats['negative_count']:,}")
print(f"   Avg forward return: {stats['avg_forward_return']:.2%}")
print(f"   Avg holding period: {stats['avg_hold_period']:.1f} days")

In [ ]:
# Visualize label distribution
spy_labeled = labeled_data['SPY']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Label distribution
label_counts = spy_labeled['Label'].value_counts()
axes[0].bar(['No Trade (0)', 'Trade (1)'], [label_counts[0], label_counts[1]], color=['red', 'green'])
axes[0].set_title('Label Distribution')
axes[0].set_ylabel('Count')
axes[0].grid(True, alpha=0.3, axis='y')

# Forward returns by label
spy_labeled.boxplot(column='ForwardReturn', by='Label', ax=axes[1])
axes[1].set_title('Forward Returns by Label')
axes[1].set_xlabel('Label')
axes[1].set_ylabel('Forward Return')
axes[1].axhline(0, color='black', linewidth=0.8)
plt.sca(axes[1])
plt.xticks([1, 2], ['No Trade (0)', 'Trade (1)'])

# Holding period distribution
axes[2].hist(spy_labeled['HoldPeriod'], bins=20, color='steelblue', edgecolor='black')
axes[2].set_title('Holding Period Distribution')
axes[2].set_xlabel('Days Held')
axes[2].set_ylabel('Frequency')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Combine All Data

In [ ]:
# Combine all tickers into single DataFrame
combined_df = pd.concat(labeled_data.values(), axis=0)
combined_df = combined_df.dropna()

print(f"\n📊 Combined Dataset:")
print(f"   Total samples: {len(combined_df):,}")
print(f"   Positive samples: {(combined_df['Label'] == 1).sum():,} ({(combined_df['Label'] == 1).mean():.1%})")
print(f"   Features: {len(feature_names)}")
print(f"   Date range: {combined_df.index.min().date()} to {combined_df.index.max().date()}")

In [ ]:
# Show correlation of top features with forward return
correlations = combined_df[feature_names + ['ForwardReturn']].corr()['ForwardReturn'].drop('ForwardReturn')
top_correlations = correlations.abs().sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 8))
top_correlations.plot(kind='barh', color='steelblue')
plt.title('Top 15 Features by Correlation with Forward Return')
plt.xlabel('Absolute Correlation')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## Next Steps

Now that we have prepared data with features and labels, the next steps are:

1. **Model Training**: Train the ensemble of gradient boosting models
2. **Walk-Forward Validation**: Test on multiple time windows
3. **Backtesting**: Simulate realistic trading with transaction costs
4. **Analysis**: Analyze performance metrics and visualize results

See the other notebooks for these steps:
- `02_model_training.ipynb`
- `03_backtesting.ipynb`
- `04_results_analysis.ipynb`